# Invoice BI Dashboard

Local Python visualization for invoice performance, payment status, clients, and overdue balances.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

CSV_FILE = 'invoices_clean.csv'
df = pd.read_csv(CSV_FILE, parse_dates=['date', 'due_date'])

df['line_total'] = pd.to_numeric(df['line_total'], errors='coerce')
df['month_start'] = df['date'].dt.to_period('M').dt.to_timestamp()

# Keep invoice counts distinct because the source contains one row per line item.
invoice_level = (
    df.groupby('invoice_no', as_index=False)
      .agg(invoice_date=('date', 'min'), status=('status', 'first'), client=('client', 'first'), invoice_total=('line_total', 'sum'))
)

monthly_revenue = df.groupby('month_start', as_index=False)['line_total'].sum()
status_revenue = invoice_level.groupby('status', as_index=False)['invoice_total'].sum().sort_values('invoice_total', ascending=False)
client_revenue = invoice_level.groupby('client', as_index=False)['invoice_total'].sum().sort_values('invoice_total', ascending=True)
overdue = invoice_level[invoice_level['status'].eq('OVERDUE')].sort_values('invoice_total', ascending=False).head(10)

total_revenue = invoice_level['invoice_total'].sum()
invoice_count = invoice_level['invoice_no'].nunique()
paid_revenue = invoice_level.loc[invoice_level['status'].eq('PAID'), 'invoice_total'].sum()
overdue_revenue = invoice_level.loc[invoice_level['status'].eq('OVERDUE'), 'invoice_total'].sum()

def format_currency(value):
    return f'Rp {value / 1_000_000:,.1f}M'

currency_formatter = FuncFormatter(lambda value, position: f'Rp {value / 1_000_000:,.0f}M')
status_colors = {
    'PAID': '#2a9d8f',
    'PARTIALLY PAID': '#e9c46a',
    'PENDING': '#457b9d',
    'OVERDUE': '#e76f51',
    'CANCELLED': '#8d99ae',
}

plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(16, 10), facecolor='#f7f8fa')
grid = fig.add_gridspec(3, 2, height_ratios=[0.8, 1.35, 1.35], hspace=0.42, wspace=0.25)
fig.suptitle('Invoice Performance Dashboard', fontsize=22, fontweight='bold', color='#1d3557', x=0.06, ha='left')
fig.text(0.06, 0.925, 'Local analysis from invoices_clean.csv', fontsize=10, color='#6c757d')

kpis = [
    ('Total invoiced', format_currency(total_revenue), '#1d3557'),
    ('Invoices', f'{invoice_count:,}', '#457b9d'),
    ('Paid revenue', format_currency(paid_revenue), '#2a9d8f'),
    ('Overdue revenue', format_currency(overdue_revenue), '#e76f51'),
]
for index, (label, value, color) in enumerate(kpis):
    axis = fig.add_subplot(grid[0, index % 2]) if index < 2 else fig.add_subplot(grid[0, index % 2])
    axis.axis('off')
    axis.text(0.02, 0.72 if index < 2 else 0.2, label.upper(), fontsize=9, color='#6c757d', transform=axis.transAxes)
    axis.text(0.02, 0.28 if index < 2 else -0.24, value, fontsize=20, fontweight='bold', color=color, transform=axis.transAxes)

# Rebuild the KPI strip as a four-column inset so all cards stay in one row.
for axis in fig.axes[:2]:
    axis.remove()
for index, (label, value, color) in enumerate(kpis):
    axis = fig.add_axes([0.06 + index * 0.235, 0.80, 0.20, 0.085])
    axis.set_facecolor('white')
    axis.axis('off')
    axis.text(0.06, 0.68, label.upper(), fontsize=8, color='#6c757d', transform=axis.transAxes)
    axis.text(0.06, 0.18, value, fontsize=17, fontweight='bold', color=color, transform=axis.transAxes)

axis = fig.add_subplot(grid[1, 0])
axis.plot(monthly_revenue['month_start'], monthly_revenue['line_total'], color='#1d3557', linewidth=2.5, marker='o', markersize=4)
axis.fill_between(monthly_revenue['month_start'], monthly_revenue['line_total'], color='#a8dadc', alpha=0.35)
axis.set_title('Monthly invoiced revenue', loc='left', fontweight='bold')
axis.set_ylabel('Revenue')
axis.yaxis.set_major_formatter(currency_formatter)
axis.tick_params(axis='x', rotation=45)

axis = fig.add_subplot(grid[1, 1])
axis.bar(status_revenue['status'], status_revenue['invoice_total'], color=[status_colors.get(status, '#6c757d') for status in status_revenue['status']])
axis.set_title('Revenue by payment status', loc='left', fontweight='bold')
axis.set_ylabel('Revenue')
axis.yaxis.set_major_formatter(currency_formatter)
axis.tick_params(axis='x', rotation=35)

axis = fig.add_subplot(grid[2, 0])
axis.barh(client_revenue['client'], client_revenue['invoice_total'], color='#457b9d')
axis.set_title('Revenue by client', loc='left', fontweight='bold')
axis.set_xlabel('Revenue')
axis.xaxis.set_major_formatter(currency_formatter)

axis = fig.add_subplot(grid[2, 1])
if overdue.empty:
    axis.text(0.5, 0.5, 'No overdue invoices', ha='center', va='center', color='#2a9d8f')
else:
    overdue_labels = overdue['invoice_no'] + ' | ' + overdue['client'].str.slice(0, 22)
    axis.barh(overdue_labels, overdue['invoice_total'], color='#e76f51')
    axis.xaxis.set_major_formatter(currency_formatter)
axis.set_title('Top overdue invoices', loc='left', fontweight='bold')
axis.set_xlabel('Invoice value')

fig.savefig('bi_dashboard.png', dpi=160, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()